In [1]:
import torch
import torch.nn as nn

class BasicBlock(nn.Module):
    expansion = 1
    
    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class Bottleneck(nn.Module):
    expansion = 4
    
    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super(ResNet, self).__init__()
        self.in_planes = 64
        
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_planes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_planes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.in_planes, planes, stride, downsample))
        self.in_planes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_planes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

def resnet18(num_classes):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

def resnet34(num_classes):
    return ResNet(BasicBlock, [3, 4, 6, 3], num_classes)

def resnet50(num_classes):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes)

def resnet101(num_classes):
    return ResNet(Bottleneck, [3, 4, 23, 3], num_classes)


In [ ]:
# Import required libraries and model architectures
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data.sampler import SubsetRandomSampler
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import StepLR
import torch.optim as optim
import pandas as pd
import gc
import seaborn as sns
from sklearn.metrics import confusion_matrix


def data_loader(data_dir, batch_size, shuffle=True, test=False):
    # Define original and augmented transforms
    original_transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    augmented_transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.RandomApply([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=45),
            transforms.RandomResizedCrop(size=512, scale=(0.8, 1.0)),  # Simulate zoom in
            transforms.ColorJitter(brightness=(0.7, 1.3), contrast=(0.3, 0.7), saturation=(0.8, 1.5)),  # Optional
            transforms.RandomAffine(degrees=0, shear=10),
            transforms.ElasticTransform(alpha=1.0),  # Elastic distortion
            transforms.RandomPerspective(distortion_scale=0.2),  # Perspective distortion
            transforms.RandomApply([
                transforms.Compose([
                    transforms.Pad(padding=20),  # Simulate zoom out by padding
                    transforms.Resize((512, 512))  # Resize back to original size
                ])
            ], p=0.5),  # 50% chance to apply zoom-out
            transforms.RandomCrop(size=(512, 512), padding=10)  # Random cropping with padding
        ], p=1),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3)),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    if not test:
        # Load training set
        original_dataset = datasets.Flowers102(root=data_dir, split='test', download=True, transform=original_transform)
        augmented_dataset = datasets.Flowers102(root=data_dir, split='test', download=True, transform=augmented_transform)
        
        # Count number of original and augmented images
        num_original = len(original_dataset)
        num_augmented = len(augmented_dataset)

        # Combine training datasets
        combined_train_dataset = torch.utils.data.ConcatDataset([original_dataset, augmented_dataset])

        # Load validation set, using flower102's own val split
        valid_dataset = datasets.Flowers102(root=data_dir, split='val', download=True, transform=original_transform)

        # Create data loaders for training and validation
        train_loader = torch.utils.data.DataLoader(combined_train_dataset, batch_size=batch_size, shuffle=shuffle, num_workers=8)
        valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=8)
        
        print(f"Number of original images: {num_original}, Number of augmented images: {num_augmented}")
        return train_loader, valid_loader, len(combined_train_dataset), len(valid_dataset)

    else:
        # Load test set
        test_loader = datasets.Flowers102(root=data_dir, split='train', download=True, transform=original_transform)
        return torch.utils.data.DataLoader(test_loader, batch_size=batch_size, shuffle=shuffle, num_workers=12), len(test_loader)




# Model training function
def train_model(model, train_loader, valid_loader, num_epoch, eval_interval=10):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

    train_losses, val_losses, train_accuracies, val_accuracies, test_accuracies = [], [], [], [],[]

    for epoch in range(num_epoch):
        model.train()
        running_train_loss, correct_train, total_train = 0.0, 0, 0

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_train_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

            del images, labels, outputs
            torch.cuda.empty_cache()
            gc.collect()

        train_loss = running_train_loss / len(train_loader)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)

        # Validation phase
        model.eval()
        val_loss, correct_val, total_val = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

                del images, labels, outputs

        val_loss /= len(valid_loader)
        val_accuracy = 100 * correct_val / total_val
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)
        scheduler.step(val_loss)

        print(f'Epoch [{epoch+1}/{num_epoch}], Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%')
        print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%')
        # Evaluate test accuracy every eval_interval epochs
        if (epoch + 1) % eval_interval == 0:
            test_accuracy, _, _ = test_model(model, test_loader)
            test_accuracies.append(test_accuracy)
            print(f"Test Accuracy after epoch {epoch+1}: {test_accuracy:.2f}%")

    return train_losses, val_losses, train_accuracies, val_accuracies, test_accuracies

# Model testing function
def test_model(model, test_loader):
    model.eval()
    correct, total = 0, 0
    class_correct = list(0. for i in range(102))  # Adjust number of classes (102 for Flowers dataset)
    class_total = list(0. for i in range(102))
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            c = (predicted == labels).squeeze()
            for i in range(len(labels)):
                label = labels[i].item()
                class_correct[label] += c[i].item()
                class_total[label] += 1

    test_accuracy = 100 * correct / total
    return test_accuracy, class_correct, class_total

# Set device and hyperparameters
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 0.0001
num_epoch = 100
batch_size = 16

# Load datasets
data_dir = './data'
print("Loading train (from test split)...")
train_loader, valid_loader, num_train, num_valid = data_loader(data_dir, batch_size=batch_size, test=False)

print("Loading test (from train split)...")
test_loader, num_test = data_loader(data_dir, batch_size=batch_size, test=True)

# Output dataset sizes
print(f"Final dataset sizes: Train: {num_train}, Validation: {num_valid}, Test: {num_test}")

resnet_models = {
    "ResNet-18": resnet18(num_classes=102),
    "ResNet-34": resnet34(num_classes=102),
    "ResNet-50": resnet50(num_classes=102),
}

# Compare model performance
def compare_resnet_models(models_dict, train_loader, valid_loader, test_loader):
    results = {}
    for model_name, model in models_dict.items():
        print(f"\nTraining {model_name}...")
        model.to(device)
        train_losses, val_losses, train_accuracies, val_accuracies, test_accuracies = train_model(model, train_loader, valid_loader, num_epoch)
        final_test_accuracy, class_correct, class_total = test_model(model, test_loader)

        # Save test accuracy and per-class accuracy to results
        results[model_name] = {
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accuracies": train_accuracies,
            "val_accuracies": val_accuracies,
            "test_accuracy": test_accuracies,
            "class_correct": class_correct,
            "class_total": class_total,
        }
        print(f"Final test Accuracy for {model_name}: {final_test_accuracy:.2f}%")

    return results


# Run comparison
results = compare_resnet_models(resnet_models, train_loader, valid_loader, test_loader)

# Plot loss and accuracy curves
for model_name, result in results.items():
    epochs = range(1, len(result["train_losses"]) + 1)

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, result["train_losses"], label='Train Loss')
    plt.plot(epochs, result["val_losses"], label='Validation Loss')
    plt.title(f'{model_name} Loss over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, result["train_accuracies"], label='Train Accuracy')
    plt.plot(epochs, result["val_accuracies"], label='Validation Accuracy')
    plt.title(f'{model_name} Accuracy over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.show()

def output_class_accuracies(results):
    for model_name, result in results.items():
        print(f"\nPer-class accuracy for {model_name}:")
        class_correct = result["class_correct"]
        class_total = result["class_total"]
        for i in range(102):  # Adjust number of classes
            if class_total[i] > 0:
                print(f'Accuracy of class {i}: {100 * class_correct[i] / class_total[i]:.2f}%')


# Output per-class accuracy
output_class_accuracies(results)

Loading train (from test split)...
Number of original images: 6149, Number of augmented images: 6149
Loading test (from train split)...
Final dataset sizes: Train: 12298, Validation: 1020, Test: 1020

Training ResNet-18...
Epoch [1/100], Train Loss: 3.5029, Train Accuracy: 18.01%
Validation Loss: 3.3315, Validation Accuracy: 18.04%
Epoch [2/100], Train Loss: 2.9502, Train Accuracy: 26.77%
Validation Loss: 3.1856, Validation Accuracy: 21.27%
Epoch [3/100], Train Loss: 2.6776, Train Accuracy: 32.62%
Validation Loss: 2.8717, Validation Accuracy: 28.82%
Epoch [4/100], Train Loss: 2.4232, Train Accuracy: 38.05%
Validation Loss: 2.6979, Validation Accuracy: 31.37%
Epoch [5/100], Train Loss: 2.1839, Train Accuracy: 43.23%
Validation Loss: 2.1931, Validation Accuracy: 43.14%
Epoch [6/100], Train Loss: 1.9724, Train Accuracy: 48.64%
Validation Loss: 2.1081, Validation Accuracy: 44.31%
Epoch [7/100], Train Loss: 1.7664, Train Accuracy: 53.70%
Validation Loss: 2.0664, Validation Accuracy: 46.47%
